#  Step 5 — Curation Pipeline
curation is basically a quality-control gate.

Having an image file does not mean that it is usable.

Imagine we find:

Patients_CT/049/brain/14.jpg

We need to ask:
- Can Python actually read it?
- Is it 650×650?
- Is it blank?
- Is it duplicated?
- Does it have a label?
- If it is a mask, does the corresponding image exist?

```
                 Step 4
             SQLite manifest
                   │
                   │ "Here are the files"
                   ▼
            ┌───────────────┐
            │   CURATION    │
            └───────────────┘
                   │
        ┌──────────┴──────────┐
        ▼                     ▼
    VALIDATE               NORMALIZE
        │                     │
        │                     │
        └──────────┬──────────┘
                   ▼
           artifacts/curated/
                   │
                   ▼
              ML-ready data
``` 

In [1]:
from medimageforge.config import load_config, data_path

config = load_config()

print(config)

{'project': {'name': 'medimageforge', 'version': '0.1.0'}, 'paths': {'data_dir': 'data', 'raw_dir': 'data/Patients_CT', 'labels_csv': 'data/hemorrhage_diagnosis.csv', 'demographics_csv': 'data/patient_demographics.csv', 'checksums_file': 'data/SHA256SUMS.txt', 'artifacts_dir': 'artifacts', 'manifest_db': 'artifacts/manifest.db', 'curated_dir': 'artifacts/curated'}, 'dataset': {'patient_id_width': 3, 'windows': ['brain', 'bone'], 'mask_suffix': '_HGE_Seg', 'expected_size': [650, 650], 'mask_threshold': 128}, 'logging': {'level': 'INFO'}}


In [2]:
print("Expected image size:", config["dataset"]["expected_size"])
print("Mask threshold:", config["dataset"]["mask_threshold"])

Expected image size: [650, 650]
Mask threshold: 128


In [4]:
raw_dir = data_path(config, "raw_dir")
curated_dir = data_path(config, "curated_dir")

print("Raw:", raw_dir)
print("Curated:", curated_dir)

Raw: /home/zahra/MedImageForge/data/Patients_CT
Curated: /home/zahra/MedImageForge/artifacts/curated


In [6]:
from PIL import Image
import numpy as np

image_path = curated_dir / "049/brain/14.png"

im = Image.open(image_path)

print("Format:", im.format)
print("Size:", im.size)
print("Mode:", im.mode)

arr = np.array(im)

print("Shape:", arr.shape)
print("Min:", arr.min())
print("Max:", arr.max())
print("Std:", arr.std())

Format: PNG
Size: (650, 650)
Mode: L
Shape: (650, 650)
Min: 0
Max: 255
Std: 84.67742415820858


In [7]:
mask_path = curated_dir / "049/brain/14_mask.png"

mask = np.array(Image.open(mask_path))

print("Unique values:", np.unique(mask))
print("White pixels:", np.sum(mask == 255))

Unique values: [  0 255]
White pixels: 687
